# Water level and wind for the dynamic runs

The met-ocean forcing of the dynamic runs (08) besides the offshore waves:

1. **Water level** at the goals from CODEC (GTSM reanalysis, hourly total
   water level, 1979-2018): one CODEC station per goal, the one the goals
   JSON snapped to (`hindcast.codec`), written to
   `forcing.water_level.data_path`. The reconstruction uses it too
   (`wl="forcing"`).
2. **Wind**: ERA5 10 m wind over the case box from Earth Data Hub (key in the
   `EDH_API_KEY` environment variable), written to `forcing.wind.data_path`.

**Prerequisites:** `04_Select_Goals_by_Hindcasts.ipynb` (CODEC snaps in the
goals JSON). **Next:** `08_Dynamic_Cases_Generation.ipynb`.

In [ ]:
import glob

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from bluemath_tk.waves.hywaves.offshore import codec_at_goals, era5_wind
from utils.config import FORCING, PATHS, load_goals, resolve

## Configuration

In [ ]:
# Monthly CODEC files (total water level) on the GeoOcean cluster
codec_files = sorted(glob.glob("/lustre/geocean/DATA/Copernicus/CODEC/twl/reanalysis_waterlevel_hourly_*_v1.nc"))
wind_period = slice("1979", "2023")  # cover the offshore hindcast
wind_margin_deg = 0.5  # around the AoI

goals = load_goals()
water_level_path = resolve(FORCING["water_level"]["data_path"])
wind_path = resolve(FORCING["wind"]["data_path"])
aoi = gpd.read_file(resolve(PATHS["aoi_file"])).to_crs("EPSG:4326")
west, south, east, north = aoi.total_bounds
wind_bounds = (west - wind_margin_deg, south - wind_margin_deg, east + wind_margin_deg, north + wind_margin_deg)
print(f"CODEC files : {len(codec_files)} ({codec_files[0].split('_')[-3]} .. {codec_files[-1].split('_')[-3]})")
print(f"water level : {water_level_path}")
print(f"wind        : {wind_path} over {np.round(wind_bounds, 2)}")

## 1 · Water level at the goals (CODEC)

In [ ]:
# CODEC station of each goal: the snapped station of the goals JSON, matched by position
with xr.open_dataset(codec_files[0]) as first:
    sx = first["station_x_coordinate"].values
    sy = first["station_y_coordinate"].values
stations = np.array([
    int(np.argmin((sx - g["hindcast"]["codec"]["longitude"]) ** 2 + (sy - g["hindcast"]["codec"]["latitude"]) ** 2))
    for g in goals.values()
])
water_level = codec_at_goals(codec_files, stations, np.array([int(k) for k in goals]))
water_level.chunk({"time": -1, "goal": 4}).to_zarr(water_level_path, mode="w")
water_level

In [ ]:
wl = water_level[FORCING["water_level"].get("variable", "waterlevel")]
print("p0.1 / p50 / p99.9 per goal (m):")
print(np.round(wl.quantile([0.001, 0.5, 0.999], dim="time").values.T, 2))
fig, ax = plt.subplots(figsize=(12, 4))
wl.sel(time=wl.time[-24 * 30:]).plot.line(x="time", ax=ax, add_legend=False)
ax.set_title("CODEC water level at the goals, last month")
plt.show()

## 2 · Wind (ERA5, Earth Data Hub)

In [ ]:
wind = era5_wind(wind_bounds, time=wind_period)
wind.chunk({"time": 4320}).to_zarr(wind_path, mode="w")
wind

In [ ]:
speed = np.hypot(wind["u10"], wind["v10"])
fig, ax = plt.subplots(figsize=(8, 5))
speed.mean("time").plot(ax=ax)
aoi.boundary.plot(ax=ax, color="k")
ax.set_title(f"Mean 10 m wind speed {wind_period.start}-{wind_period.stop}")
plt.show()